# Elo Ratings
Each team gets one strength rating that updates after every game. Beating a strong team moves it more than beating a weak one. The rating before tip-off becomes a feature, so there is no leakage.

In [62]:
import pandas as pd

In [63]:
games = pd.read_csv('../data/games_2021_26.csv')

In [64]:
home = games[games['MATCHUP'].str.contains('vs.')]
away = games[games['MATCHUP'].str.contains('@')]
matchups = pd.merge(home, away, on='GAME_ID', suffixes=('_HOME', '_AWAY'))

## Put the games in order
Convert the dates from text, then sort oldest to newest. Elo must see games in the order they were played.

In [65]:
matchups['GAME_DATE_HOME'] = pd.to_datetime(matchups['GAME_DATE_HOME'])
matchups = matchups.sort_values(['GAME_DATE_HOME', 'GAME_ID']).reset_index(drop=True)

In [66]:
elo = {}
home_pre_rtg = []
away_pre_rtg = []
K = 20
prev_season = None

## Elo loop
Replay every game in date order. `elo` holds each team's current rating; new teams start at **1500**.

- **New season:** pull every rating 25% back toward 1500 (`0.75 × rating + 0.25 × 1500`) for summer roster changes.
- **Record** both ratings *before* the update, so the feature only uses earlier games.
- **Expected:** `1 / (1 + 10 ** ((away - home) / 400))`. A **400**-point gap means 10 to 1 odds; equal ratings give 0.5.
- **Update:** `change = K × (actual − expected)` with **K = 20** (FiveThirtyEight's NBA value). Home gains `change` and away loses it, so upsets move ratings more.

In [67]:
for _, row in matchups.iterrows():

    # Check if new season started. Soft reset ratings.
    if row['SEASON_ID_HOME'] != prev_season:
        for team in elo:
            elo[team] = 0.75 * elo[team] + 0.25 * 1500  
        prev_season = row['SEASON_ID_HOME']

    home_elo = elo.get(row['TEAM_ID_HOME'], 1500)
    away_elo = elo.get(row['TEAM_ID_AWAY'], 1500)

    home_pre_rtg.append(home_elo)
    away_pre_rtg.append(away_elo)

    # See markdown above for formula breakdown
    expected_home = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))

    actual = int(row['WL_HOME'] == 'W')
    change = K * (actual - expected_home)

    elo[row['TEAM_ID_HOME']] = home_elo + change
    elo[row['TEAM_ID_AWAY']] = away_elo - change


In [68]:
len(home_pre_rtg)

6140

In [69]:
len(matchups)

6140

In [70]:
matchups['ELO_HOME'] = home_pre_rtg
matchups['ELO_AWAY'] = away_pre_rtg

In [71]:
matchups['ELO_DIFF'] = (matchups['ELO_HOME'] - matchups['ELO_AWAY'])

In [72]:
sorted(elo.items(), key=lambda x: x[1], reverse=True)

[(1610612760, 1728.5249798006857),
 (1610612759, 1696.8059923190262),
 (1610612738, 1671.0305311187785),
 (1610612765, 1652.8788477502019),
 (1610612743, 1649.5783145489827),
 (1610612747, 1619.5836125223316),
 (1610612739, 1615.4164114288694),
 (1610612752, 1609.8920945317564),
 (1610612745, 1597.2575073934456),
 (1610612750, 1568.303277034419),
 (1610612737, 1568.202326722308),
 (1610612766, 1567.7831442297622),
 (1610612746, 1546.3648778724348),
 (1610612753, 1543.3094630564578),
 (1610612757, 1523.9485509899937),
 (1610612761, 1519.4449135435768),
 (1610612756, 1519.0874139774846),
 (1610612755, 1515.4905052298477),
 (1610612748, 1501.8553255669444),
 (1610612744, 1433.1702892394603),
 (1610612749, 1404.7768819214698),
 (1610612740, 1385.2387488852908),
 (1610612741, 1368.468126431526),
 (1610612742, 1363.0302703697469),
 (1610612758, 1358.6254380518048),
 (1610612754, 1332.5679457475553),
 (1610612763, 1312.4134437997513),
 (1610612751, 1292.4075323283503),
 (1610612762, 1291.6640

In [73]:
len(elo)

30

## Elo alone
Predict a home win when the home team's Elo is higher, with no model. On 2025-26 this gets 66.5%, vs 64.9% for logistic regression (different game sets, so not an exact comparison).

In [74]:
test = matchups[matchups['SEASON_ID_HOME'] == 22025]

In [75]:
accuracy = ((test['ELO_DIFF'] > 0) == (test['WL_HOME'] == 'W')).mean()
print(accuracy)

0.6653061224489796


## One Elo row per team
features.csv has one row per team per game, so split each game's Elo into a home row and an away row (GAME_ID, TEAM_ID, ELO) and stack them.

In [76]:
home_elo_df = matchups[['GAME_ID', 'TEAM_ID_HOME', 'ELO_HOME']].rename(columns={'TEAM_ID_HOME': 'TEAM_ID', 'ELO_HOME': 'ELO'})
away_elo_df = matchups[['GAME_ID', 'TEAM_ID_AWAY', 'ELO_AWAY']].rename(columns={'TEAM_ID_AWAY': 'TEAM_ID', 'ELO_AWAY': 'ELO'})

In [77]:
home_elo_df

,GAME_ID,TEAM_ID,ELO
0,22100001,1610612749,1500.000000
1,22100002,1610612747,1500.000000
2,22100003,1610612766,1500.000000
3,22100004,1610612765,1500.000000
4,22100005,1610612752,1500.000000
...,...,...,...
6135,22501196,1610612760,1744.536701
6136,22501197,1610612759,1708.829120
6137,22501198,1610612747,1616.881493
6138,22501199,1610612746,1539.132451


In [81]:
elo_df = pd.concat([home_elo_df, away_elo_df])
len(elo_df)

12280

In [82]:
features = pd.read_csv('../data/features.csv')
len(features)

12300

In [83]:
features = pd.merge(features, elo_df, on=['GAME_ID', 'TEAM_ID'], how='left')

In [85]:
features.to_csv('../data/features.csv', index=False)